# Ex3_PEM-MHE-physics
Inner MHE → block adjoint solve → parameter VJP → Adam. See the paper’s PEM–MHE framework.

With `LOAD_PRETRAINED = True`, load the pretrained parameters and evaluate the model. Set it to `False` to train. View training history with `tensorboard --logdir runs`.


In [ ]:
from utils import configure_runtime
configure_runtime()
from datetime import datetime
import numpy as np
import torch
from torch.utils.tensorboard import SummaryWriter

from data import load_data, partial_observation_model_inputs
from models import MLP_Residual, car_physics_raw_params, make_vehicle, parameter_dict
from estimation import (
    compute_mhe_estimations_multistart, initial_ekf_multistart_warmstarts, make_mhe_config,
    terminal_excluded_ekf_warmstart,
)
from identification import PEMMHE, calculate_PEM_loss, evaluate_PEM_loss
from utils import (
    ROOT, load_car_model_params, load_initialization, save_car_model_params, save_metrics,
    show_metrics, show_model,
)


## 1. Configuration

In [ ]:
EXPERIMENT = 3
NAME = "Ex3_PEM-MHE-physics"
LOAD_PRETRAINED = True
SEED = 0
system = dict(nx=3, nu=2, nw=3, N=40, T=0.04,
              E=np.eye(3), C=np.array([[1., 0., 0.], [0., 0., 1.]]),
              integration="rk4", rk4_steps=2, m=790.0, l1=1.248, l2=1.7328)
REFERENCE_DIR = ROOT / "runs" / NAME
OUTPUT_DIR = ROOT / "runs" / "new" / (NAME + "_" + datetime.now().strftime("%Y%m%d-%H%M%S"))
np.random.seed(SEED)
torch.manual_seed(SEED)
BATCH_SIZE = 32
TRAIN_FLAG = "physics"
NN_WIDTH = 32
MAX_ITERATIONS = 875
VALIDATION_INTERVAL = 5
LEARNING_RATES = dict(physics=0.002, NN=0.01, sbar=2e-5, Sigmainv=2e-4)
MIN_LEARNING_RATES = dict(physics=0.000125, NN=6.25e-4, sbar=1.25e-6, Sigmainv=1.25e-5)
PLATEAU_PATIENCE = 10
PLATEAU_COOLDOWN = 5
REFRESH_ON_LR_REDUCTION = False

system["hidden_limit"] = 3.0

# Initial lateral velocities: kinematic estimate, negative, and positive.
EKF_INITIAL_HIDDEN_STATES = (None, -system["hidden_limit"], system["hidden_limit"])


## 2. Data and model

In [ ]:
data = load_data(EXPERIMENT)
initialization = load_initialization(NAME)
residual = None
if TRAIN_FLAG == "hybrid":
    features = partial_observation_model_inputs(data["train"], system["l1"], system["l2"]).reshape(-1, 5)
    residual = MLP_Residual(
        5,
        4,
        features.mean(0),
        np.maximum(features.std(0), 1e-3),
        hidden_dim=NN_WIDTH
    )
model = make_vehicle(system, initialization, residual)
model.set_train_flag(TRAIN_FLAG)
if LOAD_PRETRAINED:
    load_car_model_params(model, REFERENCE_DIR / "model_params_opt.pth")


## 3. Inner MHE
The last measurement is reserved for the outer prediction error.

In [ ]:
config = make_mhe_config(system, model, initialization, BATCH_SIZE)
# config.f, config.theta and config.theta_values bind the dynamics to the generic solver.
problem = PEMMHE(model, config, {s: data[s] for s in ("train", "val")})
if LOAD_PRETRAINED:
    for split in ("train", "val"):
        problem.estimate_from_ekf(split, EKF_INITIAL_HIDDEN_STATES)
else:
    warmstarts = {s: terminal_excluded_ekf_warmstart(model, data[s])
                  for s in ("train", "val")}
    if EXPERIMENT == 2:
        # Initialize the simulation MHE with three EKF trajectories.
        for split in ("train", "val"):
            candidates = initial_ekf_multistart_warmstarts(
                model, data[split], np.array([0., -2.5, 0.]), np.array([0., 2.5, 0.]))
            compute_mhe_estimations_multistart(problem.observer, problem.loaders[split], candidates)
    else:
        for split in ("train", "val"):
            problem.estimate(split, warmstarts[split])


## 4. Adjoint–VJP and parameter update

In [ ]:
if not LOAD_PRETRAINED:
    OUTPUT_DIR.mkdir(parents=True)
    groups = [{"params": car_physics_raw_params(model), "name": "physics"},
              {"params": [model.sbar], "name": "sbar"},
              {"params": [model.Sigmainv_raw], "name": "Sigmainv"}]
    if TRAIN_FLAG == "hybrid":
        groups.insert(1, {"params": model.nn_residual.parameters(), "name": "NN"})
    for group in groups:
        group["lr"] = LEARNING_RATES[group["name"]]
    optimizer = torch.optim.Adam(groups, betas=(.5, .99), amsgrad=EXPERIMENT == 3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, factor=.5, patience=PLATEAU_PATIENCE, threshold=.005,
        cooldown=PLATEAU_COOLDOWN,
        min_lr=[MIN_LEARNING_RATES[g["name"]] for g in groups])
    best_val = float("inf")
    with SummaryWriter(OUTPUT_DIR) as writer:
        for step in range(MAX_ITERATIONS + 1):
            if step:
                optimizer.zero_grad(set_to_none=True)
                problem.backward()
                optimizer.step()
                train_pem = problem.estimate("train")
            else:
                train_pem = evaluate_PEM_loss(problem.loaders["train"], config.C)
            writer.add_scalar("PEM/train", train_pem, step)
            if step % VALIDATION_INTERVAL:
                continue
            val_pem = problem.estimate("val")
            old_rates = [g["lr"] for g in optimizer.param_groups]
            if step:
                scheduler.step(val_pem)
            if REFRESH_ON_LR_REDUCTION and old_rates != [g["lr"] for g in optimizer.param_groups]:
                for split in ("train", "val"):
                    seed = terminal_excluded_ekf_warmstart(model, data[split])
                    problem.estimate(split, seed)
                train_pem = evaluate_PEM_loss(problem.loaders["train"], config.C)
                val_pem = evaluate_PEM_loss(problem.loaders["val"], config.C)
            writer.add_scalar("PEM/val", val_pem, step)
            if val_pem < best_val:
                best_val = val_pem
                save_car_model_params(model, OUTPUT_DIR / "model_params_opt.pth")
                save_metrics(OUTPUT_DIR, train_pem, val_pem, step)
                np.savez_compressed(OUTPUT_DIR / "mhe_states.npz", **{
                    s: problem.loaders[s].dataset.xhats.numpy() for s in ("train", "val")})
            print(f"{step}: train={train_pem:.6e}, validation={val_pem:.6e}")


## 5. Model and prediction errors

In [ ]:
directory = REFERENCE_DIR if LOAD_PRETRAINED else OUTPUT_DIR
load_car_model_params(model, directory / "model_params_opt.pth")
model.eval()
print(f"State dimension: {system['nx']}; input dimension: {system['nu']}; horizon: {system['N']}")
show_model(directory, model, parameters=parameter_dict(model))
print("Arrival mean:", model.sbar.detach().numpy())
print("Arrival precision:", model.Sigmainv.detach().numpy())

if LOAD_PRETRAINED:
    costs = {s: evaluate_PEM_loss(problem.loaders[s], config.C) for s in ("train", "val")}
else:
    with np.load(directory / "mhe_states.npz") as saved:
        costs = {s: float(calculate_PEM_loss(
            torch.as_tensor(data[s]["y"], dtype=torch.float32),
            torch.as_tensor(saved[s], dtype=torch.float32), config.C)) for s in ("train", "val")}
show_metrics(costs["train"], costs["val"], mhe_warmstart=LOAD_PRETRAINED)


State dimension: 3; input dimension: 2; horizon: 40
Checkpoint: runs/Ex3_PEM-MHE-physics/model_params_opt.pth
CarDynamicsPytorch()
Model parameters: 23
Bf         7.0677223
Br         12.133463
Cf         1.1232573
Cr        0.66152829
Df         4054.7368
Dr         3529.6301
Ef       -0.61009181
Er       -0.38071072
Shf   -0.00010096841
Shr    0.00041422434
Svf        -1.149292
Svr       -15.810974
Cm1        3894.5857
Cm2      0.062706478
Cr0        1.3242183
Cr2         1.034951
Iz         2138.8882
Arrival mean: [13.858321   -0.05099314 -0.02593599]
Arrival precision: [4.7459848e-06 1.8298773e-03 6.7371875e-03]
Train PEM cost: 4.328645e-04
Validation PEM cost: 4.096040e-04
MHE results depend on the solver warm start. EKF/RTS warm starts are used here, so the costs may differ from those reported in the paper.
